# Problem 51: Metadata Filter

**Difficulty:** Medium | **Framework:** OpenAI Agents SDK

**Categories:** rag, tool-calling

This notebook accompanies [Agent Foundry](https://agent-foundry-pi.vercel.app/problems/metadata-filter).

## 1. Install Dependencies

In [ ]:
!pip install -q openai-agents

## 2. Set Up Your API Key

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## 3. Constraints

- Documents must have metadata fields including year and category
- The retriever must filter by metadata before or during search
- The agent must extract filter criteria from the user's natural language query
- Semantic search alone is not sufficient — metadata filtering is required


## 4. Starter Code (has a bug — fix it!)

The code below has an intentional issue. Read the problem description and fix it.

In [ ]:
from agents import Agent, Runner
from agents.tool import function_tool
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

embeddings = OpenAIEmbeddings()

documents = [
    Document(page_content="Employees must complete safety training annually.", metadata={"year": 2022, "category": "hr"}),
    Document(page_content="Remote work is allowed up to 3 days per week.", metadata={"year": 2023, "category": "hr"}),
    Document(page_content="All employees receive 20 days of PTO per year.", metadata={"year": 2024, "category": "hr"}),
    Document(page_content="Travel expenses must be submitted within 30 days.", metadata={"year": 2022, "category": "finance"}),
    Document(page_content="The company matches 401k contributions up to 6%.", metadata={"year": 2024, "category": "finance"}),
    Document(page_content="Data must be encrypted at rest and in transit.", metadata={"year": 2024, "category": "security"}),
]

vectorstore = FAISS.from_documents(documents, embeddings)

# BUG: No metadata filtering — retrieves semantically similar but wrong-year docs
retriever = vectorstore.as_retriever()

@function_tool
def search_policies(question: str) -> str:
    """Search company policies to answer a question."""
    docs = retriever.invoke(question)
    return "\n".join([doc.page_content for doc in docs])

agent = Agent(
    name="Policy Analyst",
    instructions="You are an expert on company policies. Answer questions about company policies accurately.",
    tools=[search_policies],
)

result = Runner.run_sync(agent, "What are the 2024 HR policies?")
print(result.final_output)

## 5. Your Solution

Modify the code above or write your fixed version below.

In [ ]:
# Write your solution here


## 6. Hints

1. Parse the user's query to extract metadata filters like year, category, or department
2. Use the vector store's built-in metadata filtering capabilities during retrieval
3. Combine metadata pre-filtering with semantic search for accurate results


## 7. Evaluation Criteria

Check your solution against these criteria:

- Extracts metadata filters from query
- Applies metadata filters during retrieval
- Returns correctly filtered results
